# Exercise 1 — Inside your camera, in Python
*Quality Management · Hands-on lab · Università degli Studi di Perugia*

This notebook does what the [web page](https://lollocappo.github.io/student-hub/materials/quality-management/hands-on/camera.html) does, one step at a time. Run the cells in order with **Shift + Enter**. Nothing here can break anything: if you get lost, *Runtime → Restart* and start again.

In [ ]:
# Run this cell first: it downloads the example files from the course repository.
import os, urllib.request
BASE = "https://raw.githubusercontent.com/LolloCappo/student-hub/main/materials/quality-management/hands-on/"

def fetch(path):
    """Download a file of the lab once, and return its local name."""
    name = os.path.basename(path)
    if not os.path.exists(name):
        urllib.request.urlretrieve(BASE + path, name)
    return name

PHOTO = fetch("img/test-chart.jpg")      # a synthetic photo of a printed test chart

In [ ]:
# Optional — use your own photo instead of the example.
# Remove the "#" in front of the two lines below, run the cell, and pick the file.
# from google.colab import files
# PHOTO = list(files.upload())[0]

## 1. A photo is a grid of numbers
An image is an array with one row per line of pixels, one column per pixel, and three numbers (red, green, blue) per pixel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

img = np.asarray(Image.open(PHOTO).convert("RGB"))
H, W, _ = img.shape
print(f"{W} x {H} pixels = {W * H / 1e6:.1f} megapixels, stored as {img.dtype} (0 to 255)")
print("the pixel in the middle:", img[H // 2, W // 2], "  [R, G, B]")

raw = W * H * 3                                  # bytes if every number were stored
saved = os.path.getsize(PHOTO)                   # bytes in the JPEG file
print(f"raw {raw / 1e6:.1f} MB, file {saved / 1e3:.0f} kB: compression 1:{raw / saved:.0f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(img); ax[0].set_title("the photo")
y, x = H // 2, W // 2
ax[1].imshow(img[y - 5:y + 6, x - 5:x + 6], interpolation="nearest"); ax[1].set_title("11 x 11 pixels from the middle")
for a in ax: a.axis("off")
plt.show()

## 2. Three colour channels
Each channel is a grey image of its own. On the sensor, each pixel had **one** colour filter (the Bayer pattern: one red, two green, one blue in every 2 x 2 block); the other two values were interpolated.

In [ ]:
grey = img.astype(float) @ [0.299, 0.587, 0.114]      # ITU-R BT.601 weights
fig, ax = plt.subplots(1, 4, figsize=(16, 3.4))
for a, (title, ch) in zip(ax, [("red", img[..., 0]), ("green", img[..., 1]), ("blue", img[..., 2]), ("grey", grey)]):
    a.imshow(ch, cmap="gray", vmin=0, vmax=255); a.set_title(title); a.axis("off")
plt.show()

## 3. How many grey levels?
8 bits give 256 levels. With fewer bits the smooth ramp breaks into bands. A threshold (Exercise 2) is the extreme case: 1 bit.

In [ ]:
ramp = grey[1040:1210, 90:1390] if PHOTO == "test-chart.jpg" else grey
fig, ax = plt.subplots(4, 1, figsize=(10, 5))
for a, bits in zip(ax, [8, 4, 2, 1]):
    step = 256 / 2 ** bits
    a.imshow(np.floor(ramp / step) * step + step / 2, cmap="gray", vmin=0, vmax=255, aspect="auto")
    a.set_title(f"{bits} bits = {2 ** bits} levels", fontsize=10); a.axis("off")
plt.tight_layout(); plt.show()

## 4. Resolution and aliasing
Keep one pixel in *n*. Picking a single pixel lets detail finer than two pixels come back as false patterns (**aliasing**); averaging the block blurs it instead.

In [ ]:
star = grey[110:830, 1460:2180] if PHOTO == "test-chart.jpg" else grey
fig, ax = plt.subplots(2, 3, figsize=(12, 8))
for j, n in enumerate([1, 4, 8]):
    picked = star[::n, ::n]
    h, w = (star.shape[0] // n) * n, (star.shape[1] // n) * n
    averaged = star[:h, :w].reshape(h // n, n, w // n, n).mean(axis=(1, 3))
    ax[0, j].imshow(picked, cmap="gray", interpolation="nearest"); ax[0, j].set_title(f"pick 1 in {n}")
    ax[1, j].imshow(averaged, cmap="gray", interpolation="nearest"); ax[1, j].set_title(f"average {n} x {n}")
for a in ax.ravel(): a.axis("off")
plt.show()

## 5. Noise
Choose an area that should be perfectly uniform. A smooth change of light across it is *shading*, not noise, so a plane is fitted and removed first.

In [ ]:
y0, x0, h, w = (1040, 1600, 220, 260) if PHOTO == "test-chart.jpg" else (H // 2 - 100, W // 2 - 100, 200, 200)

def detrend(z):
    """Remove the best-fitting plane a + b*x + c*y, keep the mean."""
    yy, xx = np.mgrid[:z.shape[0], :z.shape[1]]
    A = np.c_[np.ones(z.size), xx.ravel(), yy.ravel()]
    coef, *_ = np.linalg.lstsq(A, z.ravel(), rcond=None)
    return z - (A @ coef).reshape(z.shape) + z.mean()

for name, ch in [("red", img[..., 0]), ("green", img[..., 1]), ("blue", img[..., 2]), ("grey", grey)]:
    z = detrend(ch[y0:y0 + h, x0:x0 + w].astype(float))
    print(f"{name:6s} mean {z.mean():6.1f}   noise {z.std(ddof=1):5.2f}   SNR {z.mean() / z.std(ddof=1):5.0f}")

In [ ]:
z = detrend(grey[y0:y0 + h, x0:x0 + w])
N, noise = [], []
for k in [1, 2, 4, 8, 16]:
    hh, ww = (h // k) * k, (w // k) * k
    means = z[:hh, :ww].reshape(hh // k, k, ww // k, k).mean(axis=(1, 3))   # averages of k x k pixels
    N.append(k * k); noise.append(means.std(ddof=1))
N = np.array(N)
plt.figure(figsize=(6, 3.5))
plt.plot(N, noise, "o-", label="your photo")
plt.plot(N, noise[0] / np.sqrt(N), "k--", label="ideal 1/sqrt(N)")
plt.xscale("log", base=2); plt.xlabel("pixels averaged, N"); plt.ylabel("noise (grey levels)"); plt.legend(); plt.grid(alpha=0.3)
plt.show()
print(f"averaging {N[-1]} pixels cut the noise by x{noise[0] / noise[-1]:.1f}; random noise would give x{np.sqrt(N[-1]):.0f}")

## Questions
The same as on the web page: compression, colour, grey levels, the aliasing radius on the star (spokes are 2πr/36 pixels apart at radius *r*), and why averaging stops following 1/√N.